In [3]:
import os

os.environ['CDSAPI_URL'] = 'https://cds.climate.copernicus.eu/api'
os.environ['CDSAPI_KEY'] = '95adca30-6567-41d0-95bf-ddb375ff899a'

In [ ]:
import cdsapi

c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '2m_temperature',
            'surface_solar_radiation_downwards',
            '10m_u_component_of_wind',
            '10m_v_component_of_wind',
        ],
        'year': ['2025'],
        'month': ['06', '07', '08'],
        'day': [f"{d:02d}" for d in range(1, 32)],
        'time': [f"{h:02d}:00" for h in range(24)],

        'area': [41.5, -74.5, 40.5, -73.0],  # NYC box

        'format': 'netcdf'
    },
    'era5_nyc_multigrid.nc'
)

2026-03-30 19:37:02,462 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

b2c401caeb4a0c667378ef92a8319297.zip:   0%|          | 0.00/962k [00:00<?, ?B/s]

'era5_nyc_multigrid.nc'

In [ ]:
!pip install netcdf4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.1 MB/s eta 0:00:00


In [ ]:
!pip install netcdf4 h5netcdf

In [ ]:
import zipfile

with zipfile.ZipFile('era5_nyc_multigrid.nc', 'r') as zip_ref:
    zip_ref.extractall('era5_data')

In [ ]:
import os
print(os.listdir('era5_data'))


['data_stream-oper_stepType-instant.nc', 'data_stream-oper_stepType-accum.nc']


In [ ]:
import xarray as xr

ds_inst = xr.open_dataset('era5_data/data_stream-oper_stepType-instant.nc')
ds_acc  = xr.open_dataset('era5_data/data_stream-oper_stepType-accum.nc')

In [ ]:
df_inst = ds_inst.to_dataframe().reset_index()
df_acc  = ds_acc.to_dataframe().reset_index()

In [ ]:
print(df_inst.columns)
print(df_acc.columns)

Index(['valid_time', 'latitude', 'longitude', 't2m', 'u10', 'v10', 'number',
       'expver'],
      dtype='object')
Index(['valid_time', 'latitude', 'longitude', 'ssrd', 'number', 'expver'], dtype='object')


In [ ]:
import pandas as pd
df = pd.merge(
    df_inst,
    df_acc[['valid_time','latitude','longitude','ssrd']],
    on=['valid_time','latitude','longitude'],
    how='inner'
)

In [ ]:
df = df.sort_values(['latitude','longitude','valid_time'])

In [ ]:
df['ssrd'] = df.groupby(['latitude','longitude'])['ssrd'].diff()

In [ ]:
df['ssrd'] = df['ssrd'].fillna(0)

In [ ]:
import numpy as np

# Temperature: Kelvin → Celsius
df['temp_C'] = df['t2m'] - 273.15

# Wind speed
df['wind_speed'] = np.sqrt(df['u10']**2 + df['v10']**2)

In [ ]:
df = df[['time','latitude','longitude','temp_C','ssrd','wind_speed']]


In [ ]:
print(df.shape)
print(df[['latitude','longitude']].drop_duplicates().shape)

(77280, 6)
(35, 2)


In [ ]:
df.groupby(['latitude','longitude']).size().head()

latitude  longitude
40.5      -74.50       2208
          -74.25       2208
          -74.00       2208
          -73.75       2208
          -73.50       2208
dtype: int64

In [ ]:
print(df['latitude'].unique())
print(df['longitude'].unique())

[40.5  40.75 41.   41.25 41.5 ]
[-74.5  -74.25 -74.   -73.75 -73.5  -73.25 -73.  ]


In [ ]:
df.to_csv('era5_multigrid_nyc.csv', index=False)

MULTI-GRID SINDy AUTOMATION

In [3]:
pip install pysindy scikit-learn scipy matplotlib pandas

In [6]:
from google.colab import files
uploaded = files.upload()


Saving era5_multigrid_nyc.csv to era5_multigrid_nyc.csv


In [7]:
df = pd.read_csv('era5_multigrid_nyc.csv')

In [8]:
df.head(5)

,valid_time,latitude,longitude,temp_C,ssrd,wind_speed
0,2025-06-01 00:00:00,40.5,-74.5,14.493439,0.0,3.922136
1,2025-06-01 01:00:00,40.5,-74.5,14.347443,-236096.0,4.487328
2,2025-06-01 02:00:00,40.5,-74.5,13.257599,-8640.0,4.689408
3,2025-06-01 03:00:00,40.5,-74.5,12.753998,0.0,4.677678
4,2025-06-01 04:00:00,40.5,-74.5,12.287506,0.0,4.553905


In [9]:
print(df[['latitude','longitude']].drop_duplicates())

       latitude  longitude
0         40.50     -74.50
2208      40.50     -74.25
4416      40.50     -74.00
6624      40.50     -73.75
8832      40.50     -73.50
11040     40.50     -73.25
13248     40.50     -73.00
15456     40.75     -74.50
17664     40.75     -74.25
19872     40.75     -74.00
22080     40.75     -73.75
24288     40.75     -73.50
26496     40.75     -73.25
28704     40.75     -73.00
30912     41.00     -74.50
33120     41.00     -74.25
35328     41.00     -74.00
37536     41.00     -73.75
39744     41.00     -73.50
41952     41.00     -73.25
44160     41.00     -73.00
46368     41.25     -74.50
48576     41.25     -74.25
50784     41.25     -74.00
52992     41.25     -73.75
55200     41.25     -73.50
57408     41.25     -73.25
59616     41.25     -73.00
61824     41.50     -74.50
64032     41.50     -74.25
66240     41.50     -74.00
68448     41.50     -73.75
70656     41.50     -73.50
72864     41.50     -73.25
75072     41.50     -73.00


In [2]:
!pip install cdsapi

In [5]:
import cdsapi

c = cdsapi.Client()

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '2m_temperature',
            'surface_solar_radiation_downwards',
            '10m_u_component_of_wind',
            '10m_v_component_of_wind',
        ],
        'year': '2025',
        'month': '09',
        'day': [
            '01','02','03','04','05','06','07','08','09','10',
            '11','12','13','14','15','16','17','18','19','20',
            '21','22','23','24','25','26','27','28','29','30'
        ],
        'time': [f"{str(h).zfill(2)}:00" for h in range(24)],
        'area': [
            41.5,   # North
            -74.5,  # West
            40.5,   # South
            -73.0   # East
        ],
        'format': 'netcdf'
    },
    'era5_sep2025_multigrid.nc'
)

2026-04-12 04:34:35,992 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-

20427b8c081fdc88fedb7bf4d1036666.zip:   0%|          | 0.00/361k [00:00<?, ?B/s]

'era5_sep2025_multigrid.nc'

In [6]:
!pip install netcdf4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.6 MB/s eta 0:00:00


In [14]:
!pip install h5netcdf

In [8]:
!unzip /content/era5_sep2025_multigrid.nc

Archive:  /content/era5_sep2025_multigrid.nc
  inflating: data_stream-oper_stepType-instant.nc  
  inflating: data_stream-oper_stepType-accum.nc  


In [9]:
!ls

data_stream-oper_stepType-accum.nc    era5_sep2025_multigrid.nc
data_stream-oper_stepType-instant.nc  sample_data


In [10]:
import xarray as xr

ds_inst = xr.open_dataset('data_stream-oper_stepType-instant.nc', engine='h5netcdf')
ds_acc  = xr.open_dataset('data_stream-oper_stepType-accum.nc', engine='h5netcdf')

In [11]:
df_inst = ds_inst.to_dataframe().reset_index()
df_acc  = ds_acc.to_dataframe().reset_index()

In [14]:
print(df_inst.columns)
print(df_acc.columns)

Index(['valid_time', 'latitude', 'longitude', 't2m', 'u10', 'v10', 'number',
       'expver'],
      dtype='object')
Index(['valid_time', 'latitude', 'longitude', 'ssrd', 'number', 'expver'], dtype='object')


In [15]:
import pandas as pd

df = pd.merge(
    df_inst,
    df_acc[['valid_time','latitude','longitude','ssrd']],
    on=['valid_time','latitude','longitude'],
    how='inner'
)

In [17]:
import numpy as np

df['temp_C'] = df['t2m'] - 273.15

df['wind_speed'] = np.sqrt(
    df['u10']**2 + df['v10']**2
)

df.rename(columns={
    'surface_solar_radiation_downwards': 'ssrd',
    'time': 'valid_time'
}, inplace=True)

print(df.head())
print("Shape:", df.shape)

  valid_time  latitude  longitude         t2m       u10       v10  number  \
0 2025-09-01      41.5     -74.50  290.622986 -0.642548 -1.513901       0   
1 2025-09-01      41.5     -74.25  291.244080 -0.289032 -1.927963       0   
2 2025-09-01      41.5     -74.00  291.298767  0.164093 -2.195541       0   
3 2025-09-01      41.5     -73.75  290.966736  0.565460 -2.380112       0   
4 2025-09-01      41.5     -73.50  291.060486  0.211945 -1.299057       0   

  expver     ssrd     temp_C  wind_speed  
0   0001  28352.0  17.472992    1.644616  
1   0001  23744.0  18.094086    1.949508  
2   0001  19840.0  18.148773    2.201665  
3   0001  17152.0  17.816742    2.446360  
4   0001  17792.0  17.910492    1.316233  
Shape: (25200, 11)


In [18]:
print(df[['latitude','longitude']].drop_duplicates().shape)

(35, 2)


In [19]:
df.to_csv('/content/era5_sep2025_multigrid.csv', index=False)